# TrustSQL — Experiment Results

This notebook presents the evaluation results for the TrustSQL reliability pipeline on the BIRD benchmark.

**Pipeline Components:**
1. Schema Awareness — filter relevant tables/columns before prompting
2. Chain-of-Thought Reasoning — generate structural reasoning to guide SQL construction
3. Verification & Self-correction — syntax checks + LLM auto-fix for SQL errors

**Models evaluated:** GPT-4o-mini (OpenAI), Llama-3.1-8b (Meta via Groq)  
**Dataset:** BIRD benchmark dev set, 100 examples (925 simple / 464 moderate / 145 challenging)

In [1]:
import json
import time
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Load Experiment Results

In [ ]:
with open('results/results_baseline_openai_v4.json') as f:
    baseline_openai = json.load(f)

with open('results/results_full_pipeline_openai_v4.json') as f:
    pipeline_openai = json.load(f)

with open('results/results_full_pipeline_groq_v4.json') as f:
    pipeline_groq = json.load(f)

with open('results/results_comparison.json') as f:
    comparison = json.load(f)

baseline_groq = comparison['llama-3.1-8b_baseline']

with open('results/results_ablation.json') as f:
    ablation = json.load(f)

with open('results/results_robustness.json') as f:
    robustness = json.load(f)

print('Data loaded successfully.')
print(f"GPT-4o-mini  Baseline: {baseline_openai['ex_accuracy']:.0%}  |  Full Pipeline: {pipeline_openai['ex_accuracy']:.0%}")
print(f"Llama-3.1-8b Baseline: {baseline_groq['ex_accuracy']:.0%}  |  Full Pipeline: {pipeline_groq['ex_accuracy']:.0%}")

## 2. Overall Execution Accuracy: Baseline vs Full Pipeline

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

models = ['GPT-4o-mini', 'Llama-3.1-8b']
baseline_scores = [baseline_openai['ex_accuracy'], baseline_groq['ex_accuracy']]
pipeline_scores = [pipeline_openai['ex_accuracy'], pipeline_groq['ex_accuracy']]

x = np.arange(len(models))
w = 0.32

bars1 = ax.bar(x - w/2, baseline_scores, w, label='Baseline', color='#b0c4de', edgecolor='white')
bars2 = ax.bar(x + w/2, pipeline_scores, w, label='Full Pipeline', color='#2e75b6', edgecolor='white')

for bar, score in zip(bars1, baseline_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.0%}', ha='center', va='bottom', fontsize=10)
for bar, score in zip(bars2, pipeline_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Execution Accuracy (EX)')
ax.set_title('Baseline vs Full Pipeline — Execution Accuracy (n=100)')
ax.set_ylim(0, 0.45)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.legend()
plt.tight_layout()
plt.savefig('figure/fig_overall_accuracy.png', bbox_inches='tight')
plt.show()

## 3. Accuracy by Difficulty Level

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
difficulties = ['simple', 'moderate', 'challenging']
labels = ['Simple', 'Moderate', 'Challenging']
x = np.arange(len(difficulties))
w = 0.35
colors = {'baseline': '#b0c4de', 'pipeline': '#2e75b6'}

datasets = [
    ('GPT-4o-mini', baseline_openai, pipeline_openai),
    ('Llama-3.1-8b', baseline_groq, pipeline_groq),
]

for ax, (model_name, base, pipe) in zip(axes, datasets):
    base_vals = [base['by_difficulty'].get(d, 0) for d in difficulties]
    pipe_vals = [pipe['by_difficulty'].get(d, 0) for d in difficulties]

    b1 = ax.bar(x - w/2, base_vals, w, label='Baseline', color=colors['baseline'], edgecolor='white')
    b2 = ax.bar(x + w/2, pipe_vals, w, label='Full Pipeline', color=colors['pipeline'], edgecolor='white')

    for bar, val in zip(b1, base_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.0%}', ha='center', va='bottom', fontsize=9)
    for bar, val in zip(b2, pipe_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.0%}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_title(model_name)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 0.55)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.legend(fontsize=9)

axes[0].set_ylabel('Execution Accuracy (EX)')
fig.suptitle('Accuracy by Difficulty Level (n=100)', fontsize=13)
plt.tight_layout()
plt.savefig('figure/fig_by_difficulty.png', bbox_inches='tight')
plt.show()

## 4. Ablation Study — Contribution of Each Mechanism

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

labels = [r['label'] for r in ablation]
scores = [r['ex_accuracy'] for r in ablation]
baseline_val = scores[0]

bar_colors = ['#d0d0d0' if s == baseline_val else ('#2e75b6' if s == max(scores) else '#7aadcf')
              for s in scores]
bar_colors[-1] = '#1a4f80'

bars = ax.barh(labels, scores, color=bar_colors, edgecolor='white', height=0.5)

for bar, score in zip(bars, scores):
    delta = score - baseline_val
    delta_str = f'+{delta:.0%}' if delta > 0 else ('—' if delta == 0 else f'{delta:.0%}')
    ax.text(score + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.0%}  ({delta_str})', va='center', fontsize=10)

ax.axvline(baseline_val, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlim(0, 0.75)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_xlabel('Execution Accuracy (EX)')
ax.set_title('Ablation Study — Each Mechanism Individually vs Combined (n=20, GPT-4o-mini)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('figure/fig_ablation.png', bbox_inches='tight')
plt.show()

## 5. Error Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

error_order = ['correct', 'wrong_column', 'wrong_condition', 'wrong_join', 'wrong_table', 'syntax_error', 'execution_error', 'other']
error_labels = ['Correct', 'Wrong Column', 'Wrong Condition', 'Wrong Join', 'Wrong Table', 'Syntax Error', 'Exec Error', 'Other']
palette = ['#2e75b6', '#e05c5c', '#f0a030', '#7aadcf', '#a0a0a0', '#c05090', '#60b080', '#cccccc']

datasets = [
    ('GPT-4o-mini — Full Pipeline', pipeline_openai['error_counts']),
    ('Llama-3.1-8b — Full Pipeline', pipeline_groq['error_counts']),
]

for ax, (title, counts) in zip(axes, datasets):
    vals = [counts.get(e, 0) for e in error_order]
    non_zero = [(v, l, c) for v, l, c in zip(vals, error_labels, palette) if v > 0]
    v_nz, l_nz, c_nz = zip(*non_zero)

    wedges, texts, autotexts = ax.pie(
        v_nz, labels=None, colors=c_nz,
        autopct=lambda p: f'{p:.0f}%' if p > 4 else '',
        startangle=90, pctdistance=0.75,
        wedgeprops=dict(edgecolor='white', linewidth=1.5)
    )
    for at in autotexts:
        at.set_fontsize(9)

    ax.legend(wedges, [f'{l} ({v})' for l, v in zip(l_nz, v_nz)],
              loc='lower center', bbox_to_anchor=(0.5, -0.25),
              ncol=2, fontsize=8, frameon=False)
    ax.set_title(title)

fig.suptitle('Error Type Distribution — Full Pipeline (n=100)', fontsize=13)
plt.tight_layout()
plt.savefig('figure/fig_error_distribution.png', bbox_inches='tight')
plt.show()

## 6. Robustness Testing

We evaluate pipeline robustness under two types of question perturbation: paraphrase and synonym replacement (n=20, GPT-4o-mini, Full Pipeline).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

rob = robustness['summary']
categories = ['Original', 'Paraphrase', 'Synonym Replace']
values = [rob['ex_original'], rob['ex_paraphrase'], rob['ex_synonym']]
colors = ['#2e75b6', '#7aadcf', '#b0c4de']

bars = ax.bar(categories, values, color=colors, edgecolor='white', width=0.45)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.0%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Execution Accuracy (EX)')
ax.set_title(f'Robustness — Full Pipeline under Question Perturbation\n(n={rob["n"]}, GPT-4o-mini)')
ax.set_ylim(0, max(values) * 1.35)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

drop_para = rob['drop_paraphrase']
drop_syn = rob['drop_synonym']
ax.text(0.98, 0.95,
        f'Paraphrase drop: {drop_para:.0%}\nSynonym drop:    {drop_syn:.0%}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.8))

plt.tight_layout()
plt.savefig('figure/fig_robustness.png', bbox_inches='tight')
plt.show()

## 7. Efficiency & Cost — Latency Comparison

We measure average latency per query for Baseline vs Full Pipeline on a small sample (10 examples).

In [8]:
import sys
sys.path.insert(0, '.')

from data.loader import BirdLoader
from llms.openai_llm import OpenAILLM
from pipeline.schema_awareness import SchemaFilter
from pipeline.reasoning import ChainOfThoughtReasoner
from pipeline.sql_generator import SQLGenerator
from pipeline.verification import SQLVerifier

loader = BirdLoader()
examples = loader.load()[:10]
llm = OpenAILLM()

schema_filter = SchemaFilter(llm)
reasoner = ChainOfThoughtReasoner(llm)
generator = SQLGenerator(llm)

baseline_times = []
pipeline_times = []

print('Measuring latency on 10 examples...')

for ex in examples:
    full_schema = loader.get_schema(ex.db_id)
    db_path = loader.get_db_path(ex.db_id)

    t0 = time.time()
    generator.generate(ex.question, full_schema, ex.evidence, "")
    baseline_times.append(time.time() - t0)

    t0 = time.time()
    filtered = schema_filter.filter(ex.question, full_schema, ex.evidence)
    reasoning = reasoner.reason(ex.question, filtered, ex.evidence)
    sql = generator.generate(ex.question, filtered, ex.evidence, reasoning)
    verifier = SQLVerifier(db_path, llm=llm)
    verifier.get_final_sql(sql, verifier.verify(sql, filtered))
    pipeline_times.append(time.time() - t0)

avg_baseline = np.mean(baseline_times)
avg_pipeline = np.mean(pipeline_times)
overhead = avg_pipeline / avg_baseline

print(f'\nBaseline avg latency:      {avg_baseline:.2f}s')
print(f'Full Pipeline avg latency: {avg_pipeline:.2f}s')
print(f'Overhead factor:           {overhead:.1f}x')

Measuring latency on 10 examples...

Baseline avg latency:      1.85s
Full Pipeline avg latency: 5.67s
Overhead factor:           3.1x


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

modes = ['Baseline\n(SQL generation only)', 'Full Pipeline\n(Schema + CoT + Gen + Verify)']
avg_times = [avg_baseline, avg_pipeline]
bar_colors = ['#b0c4de', '#2e75b6']

bars = ax.bar(modes, avg_times, color=bar_colors, edgecolor='white', width=0.45)
for bar, t in zip(bars, avg_times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{t:.2f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Average Latency per Query (seconds)')
ax.set_title(f'Efficiency: Baseline vs Full Pipeline\n(overhead: {overhead:.1f}x, n=10, GPT-4o-mini)')
ax.set_ylim(0, max(avg_times) * 1.3)
plt.tight_layout()
plt.savefig('figure/fig_latency.png', bbox_inches='tight')
plt.show()

## 8. Summary Table

In [10]:
print('=' * 70)
print(f'{"Model":<20} {"Mode":<18} {"EX":>6} {"Simple":>8} {"Moderate":>10} {"Challenging":>13}')
print('=' * 70)

rows = [
    ('GPT-4o-mini',  'Baseline',      baseline_openai),
    ('GPT-4o-mini',  'Full Pipeline', pipeline_openai),
    ('Llama-3.1-8b', 'Baseline',      baseline_groq),
    ('Llama-3.1-8b', 'Full Pipeline', pipeline_groq),
]

for model, mode, data in rows:
    ex = data['ex_accuracy']
    s  = data['by_difficulty'].get('simple', 0)
    m  = data['by_difficulty'].get('moderate', 0)
    c  = data['by_difficulty'].get('challenging', 0)
    print(f'{model:<20} {mode:<18} {ex:>6.0%} {s:>8.1%} {m:>10.1%} {c:>13.1%}')

print('=' * 70)
print()
print('Ablation Study (n=20, GPT-4o-mini):')
print('-' * 45)
for r in ablation:
    delta = r['ex_accuracy'] - ablation[0]['ex_accuracy']
    delta_str = f'+{delta:.0%}' if delta > 0 else ('—' if delta == 0 else f'{delta:.0%}')
    print(f"  {r['label']:<30} {r['ex_accuracy']:>5.0%}  ({delta_str})")
print('-' * 45)
print()
rob = robustness['summary']
print('Robustness (n=20, GPT-4o-mini, Full Pipeline):')
print('-' * 45)
print(f'  Original:        {rob["ex_original"]:.0%}')
print(f'  Paraphrase:      {rob["ex_paraphrase"]:.0%}  (drop: {rob["drop_paraphrase"]:.0%})')
print(f'  Synonym Replace: {rob["ex_synonym"]:.0%}  (drop: {rob["drop_synonym"]:.0%})')
print('-' * 45)

Model                Mode                   EX   Simple   Moderate   Challenging
GPT-4o-mini          Baseline              24%    30.5%      11.4%         33.3%
GPT-4o-mini          Full Pipeline         26%    35.6%      14.3%          0.0%
Llama-3.1-8b         Baseline              17%    28.8%       0.0%          0.0%
Llama-3.1-8b         Full Pipeline         19%    25.4%      11.4%          0.0%

Ablation Study (n=20, GPT-4o-mini):
---------------------------------------------
  Baseline (no mechanisms)         20%  (—)
  Schema only                      50%  (+30%)
  CoT only                         40%  (+20%)
  Verification only                20%  (—)
  Full Pipeline (all three)        45%  (+25%)
---------------------------------------------

Robustness (n=20, GPT-4o-mini, Full Pipeline):
---------------------------------------------
  Original:        40%
  Paraphrase:      30%  (drop: 10%)
  Synonym Replace: 30%  (drop: 10%)
---------------------------------------------
